In [3]:
pwd

'/mnt/home/al2644/research/projects/perturb-r/notebook/math'

In [5]:
cd /mnt/home/al2644/research/projects/perturb-r

/mnt/home/al2644/research/projects/perturb-r


In [6]:
import random
from fractions import Fraction
from typing import List, Tuple
import pandas as pd
from datasets import Dataset, DatasetDict

import pandas as pd 
import os 

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset

import seaborn as sns

from datasets import load_dataset, load_from_disk
import numpy as np
import json

# Reasoning Benchmark Eval

In [161]:
root = "./results/math500amc23/benchmark/"
for fname in os.listdir(root):
    df = pd.read_pickle(os.path.join(root, fname))
    model = fname.replace(".pickle", "")
    acc = df[["model_is_correct"]].mean()
    print("Model: ", model, " Acc: ", acc)

Model:  Qwen2.5-3B-math8k-AM-distill-step100  Acc:  model_is_correct    0.451389
dtype: float64
Model:  Qwen2.5-3B-math8k-AM-distill-step200  Acc:  model_is_correct    0.502546
dtype: float64
Model:  Qwen2.5-3B-math8k-AM-distill-step300  Acc:  model_is_correct    0.576852
dtype: float64
Model:  Qwen2.5-3B-math8k-AM-distill-step400  Acc:  model_is_correct    0.621065
dtype: float64
Model:  Qwen2.5-3B-math8k-AM-distill-step500  Acc:  model_is_correct    0.608333
dtype: float64
Model:  Qwen2.5-3B-math8k-AM-distill-step550  Acc:  model_is_correct    0.618981
dtype: float64
Model:  Qwen2.5-3B-math8k-AM-distill-step600  Acc:  model_is_correct    0.627546
dtype: float64
Model:  Qwen2.5-3B-math8k-QwQ-distill-step100  Acc:  model_is_correct    0.465046
dtype: float64
Model:  Qwen2.5-3B-math8k-QwQ-distill-step200  Acc:  model_is_correct    0.451852
dtype: float64
Model:  Qwen2.5-3B-math8k-QwQ-distill-step300  Acc:  model_is_correct    0.525694
dtype: float64
Model:  Qwen2.5-3B-math8k-QwQ-distill

In [439]:
def compute_acc (in_df):
    stats = in_df.groupby("source")[["model_is_correct"]].mean()
    print(stats)
    print("avg: ", stats['model_is_correct'].mean())

In [153]:
df = pd.read_pickle(os.path.join(root, "QwQ-32B.pickle"))

In [154]:
stats = df.groupby("problem")[["model_is_correct"]].sum().rename(columns = {"model_is_correct": "solve_n"}).reset_index()

In [156]:
stats["solve_n"].value_counts()

solve_n
8.0    319
0.0     60
7.0     27
2.0     14
5.0     13
6.0     13
1.0     10
3.0      6
4.0      4
Name: count, dtype: int64

In [83]:
from datasets import load_dataset
train_ds = load_dataset("parquet", data_files="https://huggingface.co/datasets/hkust-nlp/SimpleRL-Zoo-Data/resolve/main/simplelr_qwen_level3to5/train.parquet")["train"]

# Corrupted Numbers

In [64]:
root = "./results/allmath/corrupt_numbers_fixed_window//"
for fname in os.listdir(root):
    if 'model_judge' not in fname:
        print(fname)
        df = pd.read_pickle(os.path.join(root, fname))
        print(f"Accuracy: {df['model_is_correct'].mean()}")

AM-Distill-Qwen-32B.pickle
Accuracy: 0.8368
DeepMath-1.5B.pickle
Accuracy: 0.5272
DeepMath-Zero-7B.pickle
Accuracy: 0.6824
DeepScaleR-1.5B-Preview.pickle
Accuracy: 0.4832
LIMO-Qwen-32B.pickle
Accuracy: 0.5664
OpenThinker3-1.5B.pickle
Accuracy: 0.608
OpenThinker3-7B.pickle
Accuracy: 0.7392
QwQ-32B.pickle
Accuracy: 0.824
Qwen3-1.7B.pickle
Accuracy: 0.7536
Qwen3-30B-A3B.pickle
Accuracy: 0.8392
Qwen3-32B.pickle
Accuracy: 0.8248
Qwen3-8B.pickle
Accuracy: 0.8168
R1-0528-Qwen3-8B.pickle
Accuracy: 0.7496
R1-Distill-Qwen-1.5B.pickle
Accuracy: 0.3896
R1-Distill-Qwen-32B.pickle
Accuracy: 0.6688
R1-Distill-Qwen-7B.pickle
Accuracy: 0.4855769230769231


In [70]:
df = pd.read_pickle(os.path.join(root, 'OpenThinker3-7B.pickle'))

example = df[df['model_is_correct']==False].sample(n=1).iloc[0]
gt = example['solution']
res = example['post_corruption_response']
prompt = example['prompt']

In [71]:
df.groupby("solve_n")["model_is_correct"].mean()

solve_n
1    0.476923
2    0.323077
3    0.480000
4    0.584615
5    0.600000
6    0.625000
7    0.715000
8    0.906667
Name: model_is_correct, dtype: float64

In [72]:
df['model_is_correct'].mean()

0.7392

# Teachability

In [8]:
root = "./results/allmath/teacher_guide/model_judge/"
[fname for fname in os.listdir(root) if "model_judge" not in fname]

['DeepMath-1.5B.pickle',
 'DeepMath-Zero-7B.pickle',
 'DeepScaleR-1.5B-Preview.pickle',
 'LIMO-Qwen-32B.pickle',
 'OpenThinker3-1.5B.pickle',
 'OpenThinker3-7B.pickle',
 'Qwen3-1.7B.pickle',
 'R1-Distill-Llama-8B.pickle',
 'R1-Distill-Qwen-1.5B.pickle',
 'R1-Distill-Qwen-32B.pickle',
 'R1-Distill-Qwen-7B.pickle']

In [158]:
df = pd.read_pickle(os.path.join(root, "OpenThinker3-7B.pickle"))
df = df[(df["ratio"].between(0.2, 0.6))]

In [159]:
stats = df.groupby(["ratio", "teacher"])[["model_is_correct"]].mean()
avg = stats["model_is_correct"].mean()
print("tier wise: ", avg)
stats

tier wise:  0.25814393939393937


model_is_correct
ratio teacher                               
0.2   AM-Distill-Qwen-32B           0.187500
      DeepSeek-R1                   0.175000
      DeepSeek-R1-0528              0.250000
      QwQ-32B                       0.237500
      Qwen3-235B-A22B               0.000000
      Qwen3-235B-A22B-2507          0.200000
      Qwen3-30B-A3B                 0.125000
      Qwen3-32B                     0.091667
0.4   AM-Distill-Qwen-32B           0.212500
      DeepSeek-R1                   0.300000
      DeepSeek-R1-0528              0.312500
      QwQ-32B                       0.325000
      Qwen3-235B-A22B               0.000000
      Qwen3-235B-A22B-2507          0.400000
      Qwen3-30B-A3B                 0.215909
      Qwen3-32B                     0.183333
0.6   AM-Distill-Qwen-32B           0.237500
      DeepSeek-R1                   0.275000
      DeepSeek-R1-0528              0.625000
      QwQ-32B                       0.275000
      Qwen3-235B-A22B               0.312500
      Qwen3-235B-A22B-2507          0.700000
      Qwen3-30B-A3B                 0.329545
      Qwen3-32B                     0.225000

In [160]:
df = df[df['cross_tier']==True]
stats = df.groupby(["ratio", "teacher"])[["model_is_correct"]].mean()
avg = stats["model_is_correct"].mean()
print("tier wise: ", avg)
stats

tier wise:  0.2343501984126984


model_is_correct
ratio teacher                               
0.2   AM-Distill-Qwen-32B           0.208333
      DeepSeek-R1                   0.175000
      DeepSeek-R1-0528              0.285714
      QwQ-32B                       0.237500
      Qwen3-235B-A22B               0.000000
      Qwen3-235B-A22B-2507          0.000000
      Qwen3-30B-A3B                 0.125000
      Qwen3-32B                     0.080357
0.4   AM-Distill-Qwen-32B           0.236111
      DeepSeek-R1                   0.300000
      DeepSeek-R1-0528              0.303571
      QwQ-32B                       0.325000
      Qwen3-235B-A22B               0.000000
      Qwen3-235B-A22B-2507          0.125000
      Qwen3-30B-A3B                 0.237500
      Qwen3-32B                     0.196429
0.6   AM-Distill-Qwen-32B           0.263889
      DeepSeek-R1                   0.275000
      DeepSeek-R1-0528              0.696429
      QwQ-32B                       0.275000
      Qwen3-235B-A22B               0.312500
      Qwen3-235B-A22B-2507          0.375000
      Qwen3-30B-A3B                 0.350000
      Qwen3-32B                     0.241071

# Distractor Injection

In [162]:
root = './results/allmath/inject_distractor_old/model_judge/'
[fname for fname in os.listdir(root) if "model_judge" not in fname]

['AM-Distill-Qwen-32B.pickle',
 'DeepMath-1.5B.pickle',
 'DeepMath-Zero-7B.pickle',
 'DeepScaleR-1.5B-Preview.pickle',
 'LIMO-Qwen-32B.pickle',
 'OpenThinker3-1.5B.pickle',
 'OpenThinker3-7B.pickle',
 'QwQ-32B.pickle',
 'Qwen3-1.7B.pickle',
 'Qwen3-30B-A3B.pickle',
 'Qwen3-32B.pickle',
 'Qwen3-8B.pickle',
 'R1-0528-Qwen3-8B.pickle',
 'R1-Distill-Qwen-1.5B.pickle',
 'R1-Distill-Qwen-32B.pickle',
 'R1-Distill-Qwen-7B.pickle']

In [41]:
for fname in [fname for fname in os.listdir(root) if "model_judge" not in fname]:
    print(fname)
    df = pd.read_pickle(os.path.join(root, fname))
    try:
        df["model_is_correct"] = df["model_is_correct"].map({True: 1.0, False: 0.0})
#         df = df.rename(columns = {"solution_is_correct": 'model_is_correct'})
#         df = df.drop(columns = ['distractor_solution_is_correct'])
        df.to_pickle(os.path.join(root, fname))
    except:
        print("Done with ", fname)

AM-Distill-Qwen-32B.pickle
DeepMath-1.5B.pickle
DeepMath-Zero-7B.pickle
DeepScaleR-1.5B-Preview.pickle
LIMO-Qwen-32B.pickle
OpenThinker3-1.5B.pickle
OpenThinker3-7B.pickle
QwQ-32B.pickle
Qwen3-1.7B.pickle
Qwen3-30B-A3B.pickle
Qwen3-32B.pickle
Qwen3-8B.pickle
R1-0528-Qwen3-8B.pickle
R1-Distill-Qwen-1.5B.pickle
R1-Distill-Qwen-32B.pickle
R1-Distill-Qwen-7B.pickle


In [65]:
def focus_analysis(df):    
    df = df[df['distractor_ratio'] == 0.2]
    nv_series = df.groupby(["original_ratio"])[["model_is_correct"]].mean()
    nv_avg = df['model_is_correct'].mean()

    df["w_correct"] = df["model_is_correct"] / df["solve_n"]
    w_avg_series = df.groupby(["original_ratio"])[["w_correct"]].mean()
    w_avg = df["w_correct"].mean()

    return nv_avg, w_avg, nv_series, w_avg_series

In [80]:
df = pd.read_pickle(os.path.join(root, "AM-Distill-Qwen-32B.pickle"))
focus_analysis(df)

/tmp/ipykernel_3470702/3258258445.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["w_correct"] = df["model_is_correct"] / df["solve_n"]


(np.float64(0.2528),
 np.float64(0.042475238095238094),
                 model_is_correct
 original_ratio                  
 0.0                        0.196
 0.2                        0.268
 0.4                        0.296
 0.6                        0.264
 0.8                        0.240,
                 w_correct
 original_ratio           
 0.0              0.031686
 0.2              0.048448
 0.4              0.047243
 0.6              0.045000
 0.8              0.040000)

# Playground

In [396]:
import pandas as pd

# -------------------------------------------------
# 1) Avg from math benchmarks table
# -------------------------------------------------
benchmarks_data = [
    ("R1-Distill-Qwen-1.5B", 41.5),
    ("OpenThinker3-1.5B", 56.8),
    ("DeepScaleR-1.5B", 50.0),
    ("DeepMath-1.5B", 49.0),
    ("Qwen3-1.7B", 54.9),

    ("R1-Distill-Qwen-7B", 60.4),
    ("OpenThinker3-7B", 70.9),
    ("DeepMath-Zero-7B", 39.1),
    ("Qwen3-8B", 77.3),
    ("R1-0528-Qwen3-8B", 80.7),

    ("R1-Distill-Qwen-32B", 69.1),
    ("QwQ-32B", 78.6),
    ("LIMO-32B", 64.0),
    ("AM-Thinking-v1-32B", 82.9),
    ("Qwen3-30B-A3B", 79.5),
    ("Qwen3-32B", 80.0),
]

df_bench = pd.DataFrame(benchmarks_data, columns=["model_name", "avg"])


# -------------------------------------------------
# 2) Avg from corruption-distribution table
# -------------------------------------------------
corrupt_data = [
    ("R1-Distill-Qwen-1.5B", 39.0),
    ("OpenThinker3-1.5B", 60.8),
    ("DeepScaleR-1.5B-Preview", 48.3),
    ("DeepMath-1.5B", 52.7),
    ("Qwen3-1.7B", 75.4),

    ("R1-Distill-Qwen-7B", 48.6),
    ("OpenThinker3-7B", 60.8),
    ("DeepMath-Zero-7B", 68.3),
    ("Qwen3-8B", 81.7),
    ("R1-0528-Qwen3-8B", 75.0),

    ("R1-Distill-Qwen-32B", 66.9),
    ("QwQ-32B", 82.4),
    ("LIMO-32B", 56.6),
    ("AM-Thinking-v1-32B", 83.7),
    ("Qwen3-30B-A3B", 83.9),
    ("Qwen3-32B", 82.5),
]

df_corrsrc = pd.DataFrame(corrupt_data, columns=["model_name", "avg"])


# -------------------------------------------------
# 3) (Optional) Standardize names so we can merge
# -------------------------------------------------
# Canonicalization: collapse "-Preview" to nothing so it matches the benchmarks table.
def canonicalize(name: str) -> str:
    return name.replace("-Preview", "")

df_bench["model_key"] = df_bench["model_name"].map(canonicalize)
df_corrsrc["model_key"] = df_corrsrc["model_name"].map(canonicalize)

# Inner join on canonical name
df_merged = pd.merge(
    df_bench, df_corrsrc, on="model_key", suffixes=("_bench", "_corr")
)

# -------------------------------------------------
# 4) Correlations
# -------------------------------------------------
pearson_r = df_merged["avg_bench"].corr(df_merged["avg_corr"], method="pearson")
spearman_rho = df_merged["avg_bench"].corr(df_merged["avg_corr"], method="spearman")

print("Benchmarks dataframe:\n", df_bench, "\n")
print("Corruption dataframe:\n", df_corrsrc, "\n")
print("Merged dataframe (overlap):\n", df_merged[["model_key","avg_bench","avg_corr"]], "\n")
print(f"Pearson r = {pearson_r:.4f}")
print(f"Spearman rho = {spearman_rho:.4f}")


Benchmarks dataframe:
               model_name   avg             model_key
0   R1-Distill-Qwen-1.5B  41.5  R1-Distill-Qwen-1.5B
1      OpenThinker3-1.5B  56.8     OpenThinker3-1.5B
2        DeepScaleR-1.5B  50.0       DeepScaleR-1.5B
3          DeepMath-1.5B  49.0         DeepMath-1.5B
4             Qwen3-1.7B  54.9            Qwen3-1.7B
5     R1-Distill-Qwen-7B  60.4    R1-Distill-Qwen-7B
6        OpenThinker3-7B  70.9       OpenThinker3-7B
7       DeepMath-Zero-7B  39.1      DeepMath-Zero-7B
8               Qwen3-8B  77.3              Qwen3-8B
9       R1-0528-Qwen3-8B  80.7      R1-0528-Qwen3-8B
10   R1-Distill-Qwen-32B  69.1   R1-Distill-Qwen-32B
11               QwQ-32B  78.6               QwQ-32B
12              LIMO-32B  64.0              LIMO-32B
13    AM-Thinking-v1-32B  82.9    AM-Thinking-v1-32B
14         Qwen3-30B-A3B  79.5         Qwen3-30B-A3B
15             Qwen3-32B  80.0             Qwen3-32B 

Corruption dataframe:
                  model_name   avg             model

In [ ]:
>>> from transformers import AutoModelForCausalLM
>>> model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b")